# Real Estate Agent — Multi-turn Chat Client
Uses `invoke_agent_runtime` with `runtimeSessionId` for conversation memory.

In [ ]:
import boto3
import json
from IPython.display import Markdown, display

# Load agent ARN saved by deploy.py
with open('agent_config.json') as f:
    cfg = json.load(f)

agent_arn = cfg['agent_arn']
region    = cfg['region']

print(f'Agent ARN: {agent_arn}')
print(f'Region:    {region}')

agentcore_client   = boto3.client('bedrock-agentcore', region_name=region)
runtime_session_id = None

def parse_response(boto3_response):
    if 'text/event-stream' in boto3_response.get('contentType', ''):
        content = []
        for line in boto3_response['response'].iter_lines(chunk_size=1):
            if line:
                line = line.decode('utf-8')
                if line.startswith('data: '):
                    content.append(line[6:])
        return '\n'.join(content)
    else:
        # Concatenate ALL chunks before parsing — response can span multiple events
        raw = b''.join(event for event in boto3_response.get('response', []))
        result = json.loads(raw.decode('utf-8'))
        return result.get('response', result) if isinstance(result, dict) else result

def ask(prompt):
    global runtime_session_id

    kwargs = {
        'agentRuntimeArn': agent_arn,
        'qualifier': 'DEFAULT',
        'payload': json.dumps({'prompt': prompt}),
    }

    # Turn 2+ — pass session ID back to continue the conversation
    if runtime_session_id:
        kwargs['runtimeSessionId'] = runtime_session_id

    response = agentcore_client.invoke_agent_runtime(**kwargs)

    # Capture session ID from first response
    if not runtime_session_id:
        runtime_session_id = response.get('runtimeSessionId')
        print(f'Session started: {runtime_session_id}')

    result = parse_response(response)
    display(Markdown(f'**You:** {prompt}\n\n**Agent:** {result}'))
    return result

def new_session():
    global runtime_session_id
    runtime_session_id = None
    print('Session cleared — next ask() starts a new conversation.')

print('\nClient ready. Call ask("your question") to start.')

## Multi-turn conversation

In [ ]:
# Turn 1
ask('Which agent has the most leads?')

In [ ]:
# Turn 2 — agent remembers who "they" is from turn 1
ask('How many credits did they spend?')

In [ ]:
# Turn 3 — still in context
ask('What is their conversion rate and which community do most of their leads come from?')

In [ ]:
# Start a fresh conversation
new_session()
ask('How many total leads do we have?')